## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install lightgbm xgboost -q

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             median_absolute_error, mean_absolute_percentage_error,
                             explained_variance_score)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RAW  = "/content/drive/MyDrive/Machine_learning_project/Data/Raw"
PROC = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"
SRC  = "/content/drive/MyDrive/Machine_learning_project/src"

## Load the feature bundle

In [ ]:
b = np.load(f"{PROC}/feature_bundle.npz", allow_pickle=True)

X_tab, X_text, X_img = b["X_tab"], b["X_text"], b["X_img"]
all_ids   = b["all_ids"]
train_ids, test_ids = b["train_ids"], b["test_ids"]
y_train,   y_test   = b["y_train"],  b["y_test"]

row_of = {i: k for k, i in enumerate(all_ids)}
rows   = lambda ids: [row_of[i] for i in ids]
tr_rows, te_rows = rows(train_ids), rows(test_ids)

print("tabular:", X_tab.shape, "| text:", X_text.shape, "| image:", X_img.shape)

## The 7 feature sets and the 7 models

In [ ]:
FEATURES = {
    "tabular":            X_tab,
    "text":               X_text,
    "image":              X_img,
    "tabular+text":       np.hstack([X_tab, X_text]),
    "tabular+image":      np.hstack([X_tab, X_img]),
    "text+image":         np.hstack([X_text, X_img]),
    "tabular+text+image": np.hstack([X_tab, X_text, X_img]),
}

def make_models():
    return {
        "Dummy":        DummyRegressor(strategy="mean"),
        "Ridge":        Ridge(alpha=1.0),
        "kNN":          KNeighborsRegressor(n_neighbors=15),
        "RandomForest": RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42),
        "XGBoost":      XGBRegressor(n_estimators=800, max_depth=6, learning_rate=0.05,
                                     subsample=0.8, colsample_bytree=0.8,
                                     tree_method="hist", n_jobs=-1, random_state=42),
        "LightGBM":     LGBMRegressor(n_estimators=800, max_depth=6, learning_rate=0.05,
                                      subsample=0.8, colsample_bytree=0.8,
                                      n_jobs=-1, random_state=42, verbose=-1),
        "MLP":          "torch",     # handled separately (see next cell)
    }

# these are NOT scale-invariant, so they get standardised features
SCALE_SENSITIVE = {"Ridge", "kNN", "MLP"}

for k, v in FEATURES.items():
    print(f"{k:<20} {v.shape}")

## The MLP (PyTorch)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def train_mlp(Xtr_np, ytr_np, Xte_np, seed=42, epochs=250):
    """3-layer MLP. Returns test predictions."""
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(seed)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32, device=dev)
    Xte = torch.tensor(Xte_np, dtype=torch.float32, device=dev)
    ytr = torch.tensor(ytr_np, dtype=torch.float32, device=dev).view(-1, 1)

    layers, d = [], Xtr.shape[1]
    for h in (256, 128, 64):
        layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]
        d = h
    layers.append(nn.Linear(d, 1))
    net = nn.Sequential(*layers).to(dev)

    opt  = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.MSELoss()
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)

    net.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()             # gradients accumulate by default!
            crit(net(xb), yb).backward()
            opt.step()

    net.eval()
    with torch.no_grad():
        return net(Xte).cpu().numpy().ravel()

print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## Metrics

Every metric on **both scales**:

- **log scale** — what the model actually optimises; use this to compare models.
- **dollar scale** — what a human understands; use this to describe the error.

| Metric | Meaning |
|---|---|
| R2 | fraction of variance explained (1.0 = perfect, 0.0 = no better than the mean) |
| ExplVar | like R2 but ignores systematic bias |
| RMSE | root mean squared error — punishes big misses hard |
| MAE | mean absolute error — the *typical* miss |
| MedAE | median absolute error — robust to a few terrible predictions |
| MAPE | mean absolute percentage error |

In [ ]:
def all_metrics(y_true_log, y_pred_log):
    yt, yp = np.exp(y_true_log), np.exp(y_pred_log)     # back to dollars
    return {
        "R2":        r2_score(y_true_log, y_pred_log),
        "ExplVar":   explained_variance_score(y_true_log, y_pred_log),
        "RMSE_log":  np.sqrt(mean_squared_error(y_true_log, y_pred_log)),
        "MAE_log":   mean_absolute_error(y_true_log, y_pred_log),
        "MedAE_log": median_absolute_error(y_true_log, y_pred_log),
        "RMSE_usd":  np.sqrt(mean_squared_error(yt, yp)),
        "MAE_usd":   mean_absolute_error(yt, yp),
        "MedAE_usd": median_absolute_error(yt, yp),
        "MAPE":      mean_absolute_percentage_error(yt, yp),
    }

## Run the grid



In [ ]:
results, predictions = [], {}

for feat_name, X in FEATURES.items():
    # scaled copy for the scale-sensitive models — FIT ON TRAIN ONLY
    sc = StandardScaler().fit(X[tr_rows])
    Xs = sc.transform(X)

    for model_name, model in make_models().items():
        use = Xs if model_name in SCALE_SENSITIVE else X
        Xtr, Xte = use[tr_rows], use[te_rows]

        t0 = time.time()
        if model_name == "MLP":
            pred = train_mlp(Xtr, y_train, Xte)
        else:
            model.fit(Xtr, y_train)
            pred = model.predict(Xte)
        elapsed = time.time() - t0

        m = all_metrics(y_test, pred)
        results.append({
            "features": feat_name,
            "model": model_name,
            "n_features": X.shape[1],
            "fit_seconds": round(elapsed, 1),
            **{k: round(v, 4) for k, v in m.items()},
        })
        predictions[f"{feat_name}|{model_name}"] = pred
        print(f"{feat_name:<20} {model_name:<14} R2={m['R2']:6.3f}  ({elapsed:5.1f}s)")

results = pd.DataFrame(results).sort_values("R2", ascending=False)
print("\ndone —", len(results), "runs")

## Save results

In [ ]:
results.to_csv(f"{PROC}/model_results.csv", index=False)
np.savez_compressed(f"{PROC}/predictions.npz", y_test=y_test, **predictions)

print("saved model_results.csv and predictions.npz")
results.head(12)

## Quick look — the headline table

In [ ]:
pivot = results.pivot(index="features", columns="model", values="R2")
order = ["tabular", "text", "image", "tabular+text", "tabular+image",
         "text+image", "tabular+text+image"]
pivot.reindex(order)

## Trash

In [2]:
import os, sys, numpy as np, pandas as pd
from scipy import sparse
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD


from google.colab import drive
drive.mount('/content/drive', force_remount=False)

RAW  = "/content/drive/MyDrive/Machine_learning_project/Data/Raw"
PROC = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"

# ---- 1. spine: target + split ----
sys.path.append("/content/drive/MyDrive/Machine_learning_project/src")
import common
target = common.load_target()
train_ids, test_ids = common.load_split()

y = target.set_index("id")["log_price"]
y_train, y_test = y.loc[train_ids].values, y.loc[test_ids].values

# ---- 2. TEXT features (saved earlier) ----
X_tfidf = sparse.load_npz(f"{PROC}/text_features_v1.npz")     # (6158, 5002)
all_ids = np.load(f"{PROC}/text_features_v1_ids.npy")
X_emb   = np.load(f"{PROC}/text_emb_features.npy")            # (6158, 770)
X_both  = sparse.hstack([X_tfidf, sparse.csr_matrix(X_emb)]).tocsr()   # (6158, 5772)

# row helper (ids -> row positions)
row_of = {i: k for k, i in enumerate(all_ids)}
def rows(ids): return [row_of[i] for i in ids]
tr_rows, te_rows = rows(train_ids), rows(test_ids)

# ---- 3. IMAGE + TABULAR (teammates' files) ----
img_tr = pd.read_csv(f"{PROC}/train_embeddings.csv")
img_te = pd.read_csv(f"{PROC}/test_embeddings.csv")
tab_tr = pd.read_csv(f"{PROC}/tabular_train.csv")
tab_te = pd.read_csv(f"{PROC}/tabular_test.csv")

img_all = pd.concat([img_tr, img_te]).set_index("id")
tab_all = pd.concat([tab_tr, tab_te]).set_index("id")
img_X   = img_all.reindex(all_ids).values.astype(np.float32)

# ---- 4. TEMPORARY tabular encoding + drop the leak ----
tab_num = tab_all.copy()
for c in ['host_is_superhost','host_has_profile_pic','host_identity_verified','instant_bookable']:
    if c in tab_num: tab_num[c] = (tab_num[c] == 't').astype(float)
for c in ['host_response_time','neighbourhood_cleansed','neighbourhood_group_cleansed',
          'property_type','room_type','host_listing_bucket']:
    if c in tab_num: tab_num[c] = tab_num[c].astype('category').cat.codes.astype(float)
if 'host_verifications' in tab_num:
    tab_num['host_verifications'] = (tab_num['host_verifications'].astype(str).str.count("'")//2).astype(float)

tab_num = tab_num.drop(columns=['estimated_revenue_l365d'])   # ⚠️ target leak
tab_clean = tab_num.reindex(all_ids).values.astype(np.float32)

# ---- 5. SVD blocks (FIT ON TRAIN ONLY) ----
text_svd_small = TruncatedSVD(n_components=100, random_state=42).fit(X_both[tr_rows]).transform(X_both)
img_svd_small  = TruncatedSVD(n_components=50,  random_state=42).fit(img_X[tr_rows]).transform(img_X)

print("tab:", tab_clean.shape, "| text:", text_svd_small.shape, "| img:", img_svd_small.shape)
print("y:", y_train.shape, y_test.shape)

Mounted at /content/drive
tab: (6158, 49) | text: (6158, 100) | img: (6158, 50)
y: (4926,) (1232,)


In [3]:
import numpy as np, pandas as pd, time
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             median_absolute_error, mean_absolute_percentage_error,
                             explained_variance_score)
from xgboost import XGBRegressor

# ---------- the 7 feature combinations ----------
FEATURES = {
    "tabular":            tab_clean,
    "text":               text_svd_small,     # 100-dim
    "image":              img_svd_small,      # 50-dim
    "tabular+text":       np.hstack([tab_clean, text_svd_small]),
    "tabular+image":      np.hstack([tab_clean, img_svd_small]),
    "text+image":         np.hstack([text_svd_small, img_svd_small]),
    "tabular+text+image": np.hstack([tab_clean, text_svd_small, img_svd_small]),
}




# ---------- the 5 models ----------

def make_models():
    return {
        "Dummy":         DummyRegressor(strategy="mean"),
        "Ridge":         Ridge(alpha=1.0),
        "kNN":           KNeighborsRegressor(n_neighbors=15),
        "RandomForest":  RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42),
        "XGBoost":       XGBRegressor(n_estimators=800, max_depth=6, learning_rate=0.05,
                                      subsample=0.8, colsample_bytree=0.8,
                                      tree_method="hist", n_jobs=-1, random_state=42),
        "LightGBM":      LGBMRegressor(n_estimators=800, max_depth=6, learning_rate=0.05,
                                       subsample=0.8, colsample_bytree=0.8,
                                       n_jobs=-1, random_state=42, verbose=-1),
        "MLP":           "torch",
    }

SCALE_SENSITIVE = {"Ridge", "kNN", "MLP"}   # Dummy doesn't care; trees don't either




def all_metrics(y_true_log, y_pred_log):
    """Every metric, on both the log scale and the dollar scale."""
    yt_d, yp_d = np.exp(y_true_log), np.exp(y_pred_log)
    return {
        # --- log scale (what the model optimizes; use for model comparison) ---
        "R2":            r2_score(y_true_log, y_pred_log),
        "ExplVar":       explained_variance_score(y_true_log, y_pred_log),
        "RMSE_log":      np.sqrt(mean_squared_error(y_true_log, y_pred_log)),
        "MAE_log":       mean_absolute_error(y_true_log, y_pred_log),
        "MedAE_log":     median_absolute_error(y_true_log, y_pred_log),
        # --- dollar scale (interpretable) ---
        "RMSE_usd":      np.sqrt(mean_squared_error(yt_d, yp_d)),
        "MAE_usd":       mean_absolute_error(yt_d, yp_d),
        "MedAE_usd":     median_absolute_error(yt_d, yp_d),
        "MAPE":          mean_absolute_percentage_error(yt_d, yp_d),
    }

In [4]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def train_mlp(Xtr_np, ytr_np, Xte_np, seed=42, epochs=250):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(seed)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32, device=dev)
    Xte = torch.tensor(Xte_np, dtype=torch.float32, device=dev)
    ytr = torch.tensor(ytr_np, dtype=torch.float32, device=dev).view(-1, 1)

    layers, d = [], Xtr.shape[1]
    for h in (256, 128, 64):
        layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]
        d = h
    layers.append(nn.Linear(d, 1))
    net = nn.Sequential(*layers).to(dev)

    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.MSELoss()
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)

    net.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            crit(net(xb), yb).backward()
            opt.step()

    net.eval()
    with torch.no_grad():
        return net(Xte).cpu().numpy().ravel()

In [5]:
rows = []

for feat_name, X in FEATURES.items():
    # scaled copy for the scale-sensitive models — FIT ON TRAIN ONLY
    sc = StandardScaler().fit(X[tr_rows])
    Xs = sc.transform(X)

    for model_name, model in make_models().items():
        needs_scaling = model_name in SCALE_SENSITIVE
        Xtr = (Xs if needs_scaling else X)[tr_rows]
        Xte = (Xs if needs_scaling else X)[te_rows]

        t0 = time.time()
        if model_name == "MLP":
            pred = train_mlp(Xtr, y_train, Xte)
        else:
            model.fit(Xtr, y_train)
            pred = model.predict(Xte)
        elapsed = time.time() - t0

        m = all_metrics(y_test, pred)
        rows.append({
            "features": feat_name,
            "model": model_name,
            "n_features": X.shape[1],
            "fit_seconds": round(elapsed, 1),
            **{k: round(v, 4) for k, v in m.items()},
        })
        print(f"{feat_name:<20} {model_name:<14} R²={m['R2']:.3f}  ({elapsed:.0f}s)")

results = pd.DataFrame(rows).sort_values("R2", ascending=False)

PROC_DIR = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"
results.to_csv(f"{PROC_DIR}/model_results.csv", index=False)
print("\nsaved model_results.csv")
results.head(10)

tabular              Dummy          R²=-0.004  (0s)
tabular              Ridge          R²=0.605  (0s)
tabular              kNN            R²=0.607  (0s)
tabular              RandomForest   R²=0.715  (47s)
tabular              XGBoost        R²=0.745  (3s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


tabular              LightGBM       R²=0.737  (1s)
tabular              MLP            R²=0.696  (40s)
text                 Dummy          R²=-0.004  (0s)
text                 Ridge          R²=0.547  (0s)
text                 kNN            R²=0.449  (0s)
text                 RandomForest   R²=0.530  (255s)
text                 XGBoost        R²=0.586  (27s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


text                 LightGBM       R²=0.589  (5s)
text                 MLP            R²=0.625  (34s)
image                Dummy          R²=-0.004  (0s)
image                Ridge          R²=0.180  (0s)
image                kNN            R²=0.129  (0s)
image                RandomForest   R²=0.250  (131s)
image                XGBoost        R²=0.250  (11s)


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


text+image           LightGBM       R²=0.595  (9s)
text+image           MLP            R²=0.583  (35s)
tabular+text+image   Dummy          R²=-0.004  (0s)
tabular+text+image   Ridge          R²=0.676  (0s)
tabular+text+image   kNN            R²=0.551  (0s)
tabular+text+image   RandomForest   R²=0.706  (432s)
tabular+text+image   XGBoost        R²=0.745  (48s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


tabular+text+image   LightGBM       R²=0.751  (8s)
tabular+text+image   MLP            R²=0.736  (35s)

saved model_results.csv


,features,model,n_features,fit_seconds,R2,ExplVar,RMSE_log,MAE_log,MedAE_log,RMSE_usd,MAE_usd,MedAE_usd,MAPE
26,tabular+text,LightGBM,149,5.4,0.7518,0.7534,0.2874,0.2049,0.1544,90.2744,42.1167,22.3398,0.1987
47,tabular+text+image,LightGBM,199,8.1,0.7511,0.7526,0.2878,0.2049,0.1508,90.1105,42.1710,21.7178,0.1994
25,tabular+text,XGBoost,149,32.6,0.7497,0.7511,0.2886,0.2060,0.1536,90.3302,42.2509,21.6719,0.2001
46,tabular+text+image,XGBoost,199,47.9,0.7453,0.7467,0.2911,0.2079,0.1537,91.4457,42.7970,21.5674,0.2026
4,tabular,XGBoost,49,3.0,0.7447,0.7460,0.2915,0.2103,0.1558,89.8468,43.1783,22.1528,0.2050
32,tabular+image,XGBoost,99,16.2,0.7433,0.7444,0.2923,0.2099,0.1547,89.9446,43.0082,21.9105,0.2062
33,tabular+image,LightGBM,99,3.4,0.7381,0.7393,0.2952,0.2144,0.1600,89.6403,43.7652,22.7270,0.2105
5,tabular,LightGBM,49,1.1,0.7370,0.7382,0.2958,0.2129,0.1594,90.6591,43.5804,21.5671,0.2090
48,tabular+text+image,MLP,199,34.8,0.7363,0.7366,0.2962,0.2168,0.1669,90.1361,44.2158,24.9103,0.2146
27,tabular+text,MLP,149,34.3,0.7274,0.7302,0.3012,0.2176,0.1685,92.1421,44.3682,23.5934,0.2120
